# 01. Stacks

**Tier:** Deep. **Why you'd use this:** A text field treats `#` as backspace. You type characters in order, and each `#` erases the previous kept character. The final visible string is what remains after every erase.

**After this lesson you can:** push kept characters onto a list; pop on each backspace; read the list as the typed result.
About 25 minutes. **Uses:** [Lists](../../course/04-collections/01-lists-lesson.ipynb), [Loops](../../course/03-control-flow/02-loops-lesson.ipynb)


## Quick review

1. What does `append` do to a list used as a stack?
2. What does `pop` remove when you call it with no index?

<details><summary>Answers</summary>

`append` adds at the end. `pop()` removes and returns the last value — last in, first out. Skim [Lists](../../course/04-collections/01-lists-lesson.ipynb) if either is unclear.
</details>


## Recognize it

You process characters left to right, and some characters undo a previous choice. The prompt talks about backspaces, undoing the last push, or "build the result with a stack".

Signals: latest unfinished work must be undone first; order of arrival matters. A look-alike that is not this pattern: counting nesting depth with an integer when you never need the stored characters themselves — a counter is enough then.


## Why it works this way

A nested rebuild for every `#` would copy the kept prefix into a new string each time. On a string of length `n` full of alternating letters and deletes, those copies grow like `1 + 2 + … + n`, which is proportional to `n²`. A stack stores the kept characters once. On a letter you append. On `#` you pop if the stack is not empty. At the end you join the stack. Each character is handled a constant number of times, so time stays proportional to `n`. Extra memory is the kept characters — at most `n`.

The stack invariant after each step is: the list holds exactly the characters that would still be visible if typing stopped now. An empty stack means a `#` does nothing; it must not raise or invent a character. That matches real editors: backspace on an empty field is a no-op.

A two-pointer walk from the end can also skip deleted characters without a stack, trading clarity for less allocation. The stack form is the one that matches how you describe the process: push what you type, pop what you erase. See [Lists](../../internals/03-lists/01-dynamic-array-lesson.ipynb) for why append and pop at the end are amortized constant time on CPython's dynamic array.
Comparing two typed strings is the natural next question: run the same stack process on each input and compare the resulting lists. That is the practice core. The single-string form here isolates the stack contract without the second pass. Editors that support undo stacks generalize the same idea: each edit is a push, and undo is a pop of a richer command object. Character backspace is the smallest useful version of that story.

One more edge: a string of only `#` characters yields the empty string. The loop never appends, and every pop is skipped. Returning `"".join(stack)` still works. That empty result is why callers compare with `==` rather than assuming a non-empty buffer.


## When to use it

Use a stack when later input undoes earlier input, or when you must match the most recent opener. Prefer a depth counter when you only care about balance of one kind of bracket and never need the stored values. Prefer two pointers from the end when memory must stay constant and you only compare final strings. Prefer a plain builder when nothing ever deletes.
 Matching openers of several kinds (`()`, `[]`, `{}`) needs a stack of characters, not a single depth counter, because the closer must match the latest opener's shape.


## Brute force, and why it's slow

Rebuild a new string for every character: copy everything so far, then drop the last character on `#`.


In [1]:
def typed_slow(text):
    copies = 0
    result = ""
    for character in text:
        if character == "#":
            result = result[:-1]
            copies += len(result)
        else:
            result = result + character
            copies += len(result)
    return result, copies


for size in (50, 100, 200):
    text = ("a#" * size)
    print(size, typed_slow(text))


50 ('', 50)
100 ('', 100)
200 ('', 200)


## Key insight

Keep a list of surviving characters. Letters append. `#` pops when the list is non-empty. The final join is the typed string.


## Worked example

Apply backspaces in `text`. `#` deletes the previous kept character. Return the final visible string.


In [2]:
def typed(text: str) -> str:
    # 1. Surviving characters, most recent at the end.
    stack: list[str] = []
    for character in text:
        # 2. Backspace removes the latest kept character when one exists.
        if character == "#":
            if stack:
                stack.pop()
        else:
            # 3. Anything else is typed as-is.
            stack.append(character)
    return "".join(stack)


print(typed("ab#c"))
print(typed("a##c"))
print(typed("#a#"))


ac
c



## Trace

`"ab#c"`:

| character | stack after step |
|---|---|
| `a` | `['a']` |
| `b` | `['a', 'b']` |
| `#` | `['a']` |
| `c` | `['a', 'c']` → join **`ac`** |


## Cost

One pass over the string: time grows with `n`. The stack holds at most `n` characters. The rebuild-on-every-step version pays about `n²` character copies on pathological inputs.

On a string that never deletes, the stack grows to `n` and joins once at the end — still linear. The quadratic rebuild pays for every prefix on every step even when `#` never appears, which is why the copy count in the slow cell climbs so fast.


## Variants

Not solved here: [Valid Parentheses](../../challenges/problems/08-valid-parentheses.ipynb), [Daily Temperatures](../../challenges/problems/23-daily-temperatures.ipynb), [Generate Parentheses](../../challenges/problems/35-generate-parentheses.ipynb).


## Common mistake

This bug is silent on some inputs and wrong on others. The code pops on `#` without checking for an empty stack, so a leading `#` raises `IndexError`. When the stack is non-empty the logic looks fine, which hides the empty case until a test hits it.


In [3]:
def typed_wrong(text):
    stack = []
    for character in text:
        if character == "#":
            stack.pop()  # crashes when the stack is empty
        else:
            stack.append(character)
    return "".join(stack)


print(typed_wrong("#a"))


IndexError: pop from empty list

The fix pops only when `stack` is non-empty. A leading `#` is then a no-op. Run `typed("#a")` and you get `"a"`.


In [4]:
print(typed("#a"))


a


## Python idioms

A list is the usual stack: `append` and `pop()` at the end. `collections.deque` also works, but for stack-only use the list form is enough and keeps the code short.


In [5]:
def typed_chars(text: str) -> list[str]:
    stack: list[str] = []
    for character in text:
        if character == "#" and stack:
            stack.pop()
        elif character != "#":
            stack.append(character)
    return stack


print(typed_chars("ab#c"))


['a', 'c']


## Recap

- A stack holds characters that are still visible.
- `#` pops; other characters push.
- Never pop an empty stack — treat that `#` as a no-op.

Next: [practice](01-backspace-practice.ipynb).
